# LeetCode #1263: Minimum Moves to Move a Box to Their Target Location

https://leetcode.com/problems/minimum-moves-to-move-a-box-to-their-target-location/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (DFS)** | Exponential | $O(m^2 n^2)$ |
| **Optimal: 0-1 BFS on (box, player) State ★** | $O(m^2 n^2)$ | $O(m^2 n^2)$ |

---

## Understanding the Methods

### Brute Force (DFS)
Recursively explore all box-push and player-move combinations without pruning. Revisits states and cannot guarantee minimum pushes.

### Optimal: 0-1 BFS on (box, player) State ★
Model state as $(box\_r, box\_c, player\_r, player\_c)$. Pushing the box costs 1; player repositioning costs 0. Use a deque (0-1 BFS): push-moves go to the back (cost 1) and would-be player moves are embedded in a BFS reachability check. The minimum number of pushes to reach the target is the answer.

**Why this is better than Brute Force:** 0-1 BFS visits each state at most once and processes states in non-decreasing cost order, guaranteeing the minimum-push solution efficiently.

**Constraints:**
* $1 \le m, n \le 20$
* Grid contains exactly one 'S' (player), one 'B' (box), one 'T' (target)
* Obstacles are '#'

## Solutions

### C#

In [ ]:
using System.Collections.Generic;
public class Solution {
    int m, n;
    char[][] g;
    public int MinPushBox(char[][] grid) {
        g = grid; m = grid.Length; n = grid[0].Length;
        int br = 0, bc = 0, pr = 0, pc = 0, tr = 0, tc = 0;
        for (int r = 0; r < m; r++)
            for (int c = 0; c < n; c++) {
                if (g[r][c] == 'B') { br = r; bc = c; }
                else if (g[r][c] == 'S') { pr = r; pc = c; }
                else if (g[r][c] == 'T') { tr = r; tc = c; }
            }
        // State: (box_r, box_c, player_r, player_c) -> min pushes
        var dist = new int[m, n, m, n];
        for (int a = 0; a < m; a++) for (int b = 0; b < n; b++)
            for (int c2 = 0; c2 < m; c2++) for (int d = 0; d < n; d++)
                dist[a, b, c2, d] = int.MaxValue;
        dist[br, bc, pr, pc] = 0;
        // Deque for 0-1 BFS: cost-0 player moves embedded in reachability check
        var deque = new LinkedList<(int, int, int, int)>();
        deque.AddFirst((br, bc, pr, pc));
        int[] dr = { -1, 1, 0, 0 }, dc = { 0, 0, -1, 1 };
        while (deque.Count > 0) {
            var (bR, bC, pR, pC) = deque.First.Value; deque.RemoveFirst();
            int pushes = dist[bR, bC, pR, pC];
            if (bR == tr && bC == tc) return pushes;
            // Try pushing box in each direction by repositioning player first
            for (int d = 0; d < 4; d++) {
                int nbR = bR + dr[d], nbC = bC + dc[d]; // new box position
                int needR = bR - dr[d], needC = bC - dc[d]; // player must be opposite side
                if (nbR < 0 || nbR >= m || nbC < 0 || nbC >= n || g[nbR][nbC] == '#') continue;
                if (needR < 0 || needR >= m || needC < 0 || needC >= n || g[needR][needC] == '#') continue;
                // Check if player can reach (needR, needC) without moving the box
                if (!CanReach(pR, pC, needR, needC, bR, bC)) continue;
                if (dist[nbR, nbC, bR, bC] > pushes + 1) {
                    dist[nbR, nbC, bR, bC] = pushes + 1;
                    deque.AddLast((nbR, nbC, bR, bC));
                }
            }
        }
        return -1;
    }
    // BFS to check if player can walk from (sr,sc) to (tr,tc) avoiding the box
    bool CanReach(int sr, int sc, int tr2, int tc2, int bR, int bC) {
        if (sr == tr2 && sc == tc2) return true;
        var q = new Queue<(int, int)>();
        q.Enqueue((sr, sc));
        var seen = new bool[m, n];
        seen[sr, sc] = true;
        int[] dr = { -1, 1, 0, 0 }, dc = { 0, 0, -1, 1 };
        while (q.Count > 0) {
            var (r, c) = q.Dequeue();
            for (int d = 0; d < 4; d++) {
                int nr = r + dr[d], nc = c + dc[d];
                if (nr < 0 || nr >= m || nc < 0 || nc >= n || seen[nr, nc] || g[nr][nc] == '#') continue;
                if (nr == bR && nc == bC) continue; // cannot walk through the box
                seen[nr, nc] = true;
                if (nr == tr2 && nc == tc2) return true;
                q.Enqueue((nr, nc));
            }
        }
        return false;
    }
}

### Python

In [ ]:
from collections import deque
class Solution:
    def min_push_box(self, grid: list[list[str]]) -> int:
        m, n = len(grid), len(grid[0])
        br = bc = pr = pc = tr = tc = 0
        for r in range(m):
            for c in range(n):
                if grid[r][c] == 'B': br, bc = r, c
                elif grid[r][c] == 'S': pr, pc = r, c
                elif grid[r][c] == 'T': tr, tc = r, c
        dirs = [(-1,0),(1,0),(0,-1),(0,1)]
        def can_reach(sr, sc, er, ec, box_r, box_c):
            # BFS to check if player can walk from start to end avoiding the box
            if sr == er and sc == ec: return True
            q = deque([(sr, sc)])
            seen = {(sr, sc)}
            while q:
                r, c = q.popleft()
                for dr, dc in dirs:
                    nr, nc = r+dr, c+dc
                    if 0<=nr<m and 0<=nc<n and (nr,nc) not in seen and grid[nr][nc]!='#' and (nr,nc)!=(box_r,box_c):
                        if nr==er and nc==ec: return True
                        seen.add((nr,nc)); q.append((nr,nc))
            return False
        # Deque for BFS: state (box_r, box_c, player_r, player_c) -> min pushes
        INF = float('inf')
        dist = [[[[INF]*n for _ in range(m)] for _ in range(n)] for _ in range(m)]
        dist[br][bc][pr][pc] = 0
        dq = deque([(br, bc, pr, pc)])
        while dq:
            bR, bC, pR, pC = dq.popleft()
            pushes = dist[bR][bC][pR][pC]
            if bR == tr and bC == tc: return pushes
            for dr, dc in dirs:
                nbR, nbC = bR+dr, bC+dc
                needR, needC = bR-dr, bC-dc  # player must stand opposite the push direction
                if not (0<=nbR<m and 0<=nbC<n) or grid[nbR][nbC]=='#': continue
                if not (0<=needR<m and 0<=needC<n) or grid[needR][needC]=='#': continue
                if not can_reach(pR, pC, needR, needC, bR, bC): continue
                if dist[nbR][nbC][bR][bC] > pushes+1:
                    dist[nbR][nbC][bR][bC] = pushes+1
                    dq.append((nbR, nbC, bR, bC))
        return -1

### Go

In [ ]:
func minPushBox(grid [][]byte) int {
    m, n := len(grid), len(grid[0])
    var br, bc, pr, pc, tr, tc int
    for r := 0; r < m; r++ {
        for c := 0; c < n; c++ {
            switch grid[r][c] {
            case 'B': br, bc = r, c
            case 'S': pr, pc = r, c
            case 'T': tr, tc = r, c
            }
        }
    }
    dirs := [][2]int{{-1,0},{1,0},{0,-1},{0,1}}
    canReach := func(sr, sc, er, ec, boxR, boxC int) bool {
        // BFS to check if player can walk from start to end avoiding the box
        if sr == er && sc == ec { return true }
        type P struct{ r, c int }
        q := []P{{sr, sc}}
        seen := map[P]bool{{sr, sc}: true}
        for len(q) > 0 {
            cur := q[0]; q = q[1:]
            for _, d := range dirs {
                nr, nc := cur.r+d[0], cur.c+d[1]
                p := P{nr, nc}
                if nr<0||nr>=m||nc<0||nc>=n||seen[p]||grid[nr][nc]=='#'||(nr==boxR&&nc==boxC) { continue }
                if nr==er && nc==ec { return true }
                seen[p] = true; q = append(q, p)
            }
        }
        return false
    }
    type State struct{ bR, bC, pR, pC int }
    INF := 1<<30
    dist := map[State]int{{br, bc, pr, pc}: 0}
    dq := []State{{br, bc, pr, pc}}
    for len(dq) > 0 {
        cur := dq[0]; dq = dq[1:]
        pushes := dist[cur]
        if cur.bR==tr && cur.bC==tc { return pushes }
        for _, d := range dirs {
            nbR, nbC := cur.bR+d[0], cur.bC+d[1]
            needR, needC := cur.bR-d[0], cur.bC-d[1]
            if nbR<0||nbR>=m||nbC<0||nbC>=n||grid[nbR][nbC]=='#' { continue }
            if needR<0||needR>=m||needC<0||needC>=n||grid[needR][needC]=='#' { continue }
            if !canReach(cur.pR, cur.pC, needR, needC, cur.bR, cur.bC) { continue }
            ns := State{nbR, nbC, cur.bR, cur.bC}
            if v, ok := dist[ns]; !ok || v > pushes+1 {
                if pushes+1 < INF { dist[ns] = pushes + 1; dq = append(dq, ns) }
            }
        }
    }
    return -1
}

### Rust

In [ ]:
use std::collections::{VecDeque, HashMap};
impl Solution {
    pub fn min_push_box(grid: Vec<Vec<char>>) -> i32 {
        let (m, n) = (grid.len(), grid[0].len());
        let (mut br, mut bc, mut pr, mut pc, mut tr, mut tc) = (0,0,0,0,0,0);
        for r in 0..m { for c in 0..n { match grid[r][c] {
            'B' => { br=r; bc=c; } 'S' => { pr=r; pc=c; } 'T' => { tr=r; tc=c; } _ => {}
        }}}
        let dirs: &[(i32,i32)] = &[(-1,0),(1,0),(0,-1),(0,1)];
        let can_reach = |sr: usize, sc: usize, er: usize, ec: usize, box_r: usize, box_c: usize| -> bool {
            if sr==er && sc==ec { return true; }
            let mut q = VecDeque::new();
            let mut seen = vec![vec![false; n]; m];
            q.push_back((sr, sc)); seen[sr][sc] = true;
            while let Some((r,c)) = q.pop_front() {
                for &(dr,dc) in dirs {
                    let nr = r as i32+dr; let nc = c as i32+dc;
                    if nr<0||nr>=m as i32||nc<0||nc>=n as i32 { continue; }
                    let (nr,nc)=(nr as usize,nc as usize);
                    if seen[nr][nc]||grid[nr][nc]=='#'||(nr==box_r&&nc==box_c) { continue; }
                    if nr==er && nc==ec { return true; }
                    seen[nr][nc]=true; q.push_back((nr,nc));
                }
            }
            false
        };
        let mut dist: HashMap<(usize,usize,usize,usize),i32> = HashMap::new();
        dist.insert((br,bc,pr,pc),0);
        let mut dq = VecDeque::new();
        dq.push_back((br,bc,pr,pc));
        while let Some((bR,bC,pR,pC)) = dq.pop_front() {
            let pushes = dist[&(bR,bC,pR,pC)];
            if bR==tr && bC==tc { return pushes; }
            for &(dr,dc) in dirs {
                let nbR=bR as i32+dr; let nbC=bC as i32+dc;
                let neeR=bR as i32-dr; let neeC=bC as i32-dc;
                if nbR<0||nbR>=m as i32||nbC<0||nbC>=n as i32 { continue; }
                if neeR<0||neeR>=m as i32||neeC<0||neeC>=n as i32 { continue; }
                let (nbR,nbC,neeR,neeC)=(nbR as usize,nbC as usize,neeR as usize,neeC as usize);
                if grid[nbR][nbC]=='#'||grid[neeR][neeC]=='#' { continue; }
                if !can_reach(pR,pC,neeR,neeC,bR,bC) { continue; }
                let ns=(nbR,nbC,bR,bC);
                if !dist.contains_key(&ns)||dist[&ns]>pushes+1 {
                    dist.insert(ns, pushes+1); dq.push_back(ns);
                }
            }
        }
        -1
    }
}

## Example Scenarios

### 1. Common Case
**Input:** 4x4 grid with player at (3,0), box at (2,0), target at (0,3), no obstacles.
BFS finds the path of minimum pushes: push box right 3 times and up 2 times, repositioning the player between pushes. Answer: **5**.

### 2. Slightly Complex
**Input:** Player and box aligned with target but an obstacle forces a detour.
Player BFS (inner can_reach) detects unreachability from one side, forcing the algorithm to try the opposite push direction even if it adds extra pushes.

### 3. Edge Case: Time Factor
**Input:** $20 \times 20$ grid (maximum size).
State space is $(20 \times 20)^2 = 160{,}000$ combined states. Each state triggers one inner BFS of at most $400$ cells — total $\approx 64{,}000{,}000$ operations, within time limits.

### 4. Edge Case: Space Factor
**Input:** $20 \times 20$ grid.
The dist map or array holds up to $(20 \cdot 20)^2 = 160{,}000$ entries; the inner BFS visited array is $20 \times 20 = 400$ booleans reused each call. Dominant space: $O(m^2 n^2)$.

### 5. Almost-Impossible but Plausible
**Input:** Box is already at target position, player is adjacent.
The deque pops the first state and immediately returns 0 pushes — no BFS steps at all. Answer: **0**.